# Connectome-shaped RL controller: arm balancing a ball on a tray

This notebook pulls the GitHub repo, downloads the FlyWire fly-brain connectome, cuts a small sub-network out of it, and trains it with PPO to control a MuJoCo robot arm that keeps a ball on a tray. An MLP of the same size is trained as a baseline.

**Kaggle settings (right panel):**
- **Internet: ON** (needed for git clone, pip and the connectome download)
- **Accelerator: None (CPU)** is recommended. The physics runs on CPU and the network is small, so a GPU barely helps, and this saves your GPU quota.
- Use **Save Version → Save & Run All (Commit)** for long runs, so it keeps going after you close the tab.

In [ ]:
# ===== Settings =====
REPO_URL = "https://github.com/nihal4/connectome-tray-balance.git"   # change if your repo name differs
BRANCH = "main"

N_NEURONS = 1000          # size of the connectome subset (1000 = ~53k connections)
TOTAL_STEPS = 1_000_000   # per run; ~1-2 h each on Kaggle CPU (check the speed printed below)
SEED = 0
RUN_MLP_BASELINE = True   # same-size MLP for comparison
RUN_SHUFFLED = False      # same edges and weights, random wiring (control for "is it the wiring?")
PUSH_PROB = 0.0           # >0 adds random pushes on the ball (harder task)
DEVICE = "cpu"            # "cuda" only if you switched the accelerator on

In [ ]:
import os, subprocess
WORK = "/kaggle/working"
REPO = f"{WORK}/connectome-tray-balance"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# torch, numpy, pandas, pyarrow, matplotlib are already on Kaggle
!pip install -q mujoco gymnasium stable-baselines3 psutil imageio imageio-ffmpeg

## 1. Connectome: download and cut a subset

In [ ]:
# If you added the parquet as a Kaggle dataset, it is used instead of downloading.
import glob, shutil
os.makedirs("data/raw", exist_ok=True)
found = glob.glob("/kaggle/input/**/Connectivity_783.parquet", recursive=True)
if found and not os.path.exists("data/raw/Connectivity_783.parquet"):
    shutil.copy(found[0], "data/raw/")
    print("using", found[0])
!python scripts/download_connectome.py --out data/raw

In [ ]:
!python scripts/build_subset.py --n {N_NEURONS}
SUBSET = f"data/subset_flywire783_N{N_NEURONS}.npz"

## 2. Health check (environment, PD controller, network speed)

In [ ]:
!python scripts/check_mac.py --subset {SUBSET}

## 3. Train

In [ ]:
common = f"--profile kaggle --subset {SUBSET} --total_steps {TOTAL_STEPS} --seed {SEED} --push_prob {PUSH_PROB} --device {DEVICE} --runs_dir {WORK}/runs"
!python -u scripts/train.py --policy connectome --wiring real {common} 2>&1 | grep --line-buffered -E "^\[|Eval num|Episode length|New best"

In [ ]:
if RUN_MLP_BASELINE:
    !python -u scripts/train.py --policy mlp {common} 2>&1 | grep --line-buffered -E "^\[|Eval num|Episode length|New best"

In [ ]:
if RUN_SHUFFLED:
    !python -u scripts/train.py --policy connectome --wiring shuffled {common} 2>&1 | grep --line-buffered -E "^\[|Eval num|Episode length|New best"

If a run stops early (time limit, session restart), rerun its cell with `--resume` added; it continues from the last checkpoint in `/kaggle/working/runs`.

## 4. Results

In [ ]:
runs = sorted(glob.glob(f"{WORK}/runs/*"))
print(runs)
!python scripts/plot_runs.py {' '.join(runs)} --out {WORK}/curves.png
from IPython.display import Image, display
display(Image(f"{WORK}/curves.png"))

In [ ]:
# Robustness tests the policies never saw in training: sensor noise, hidden ball ("blind"), heavy ball, pushes
for r in runs:
    print("\n#####", os.path.basename(r))
    !python scripts/evaluate.py --run {r} --robustness --episodes 10

In [ ]:
# Video of the connectome policy (needs offscreen OpenGL; skipped if unavailable)
os.environ["MUJOCO_GL"] = "egl"
vid = f"{WORK}/connectome_policy.mp4"
r = [x for x in runs if "connectome-real" in x][0]
!python scripts/evaluate.py --run {r} --episodes 1 --video {vid}
if os.path.exists(vid):
    from IPython.display import Video
    display(Video(vid, embed=True, width=480))
else:
    print("No video: try MUJOCO_GL='osmesa' after `!apt-get -qq install -y libosmesa6`")

In [ ]:
# Download everything (models, logs, eval results) as one zip from the Output panel
shutil.make_archive(f"{WORK}/runs_export", "zip", f"{WORK}/runs")
print("saved", f"{WORK}/runs_export.zip")